# Detección de Caries con YOLOv8
**Dataset:** Data_Caries.v1i.yolov11 (subido a Drive)  
**Modelo:** YOLOv8s (optimizado para GPU T4 gratuita)  
**Clases:** caries leve, caries moderada, caries severa, sin caries

> Antes de ejecutar: activar GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU

## 1. Instalar dependencias

In [ ]:
!pip install ultralytics -q
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH   = '/content/drive/MyDrive/proyecto-tesis'
DATASET_PATH = f'{DRIVE_PATH}/Data_Caries.v1i.yolov11'

os.makedirs(f'{DRIVE_PATH}/Resultados_Pruebas', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/models', exist_ok=True)

# Verificar que el dataset existe
if os.path.exists(DATASET_PATH):
    print('Dataset encontrado en Drive')
else:
    print('ERROR: No se encontró la carpeta Data_Caries.v1i.yolov11 en Drive')

## 3. Configurar rutas del dataset

In [ ]:
import yaml
from pathlib import Path

data_yaml = Path(DATASET_PATH) / 'data.yaml'

# Actualizar las rutas absolutas en data.yaml para que Colab las encuentre
with open(data_yaml) as f:
    data_config = yaml.safe_load(f)

data_config['train'] = f'{DATASET_PATH}/train/images'
data_config['val']   = f'{DATASET_PATH}/valid/images'
data_config['test']  = f'{DATASET_PATH}/test/images'

with open(data_yaml, 'w') as f:
    yaml.dump(data_config, f, allow_unicode=True)

print('Clases:', data_config['names'])
print('Número de clases:', data_config['nc'])
print('Rutas configuradas correctamente')

## 4. Explorar el dataset

In [ ]:
for split in ['train', 'valid', 'test']:
    img_dir = Path(DATASET_PATH) / split / 'images'
    if img_dir.exists():
        count = len(list(img_dir.glob('*')))
        print(f'{split}: {count} imágenes')

## 5. Visualizar muestras del dataset con bounding boxes

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

COLORS = ['#22c55e', '#eab308', '#ef4444', '#3b82f6']

def plot_yolo_sample(img_path, label_path, class_names):
    img = Image.open(img_path)
    w, h = img.size
    fig, ax = plt.subplots(1, figsize=(8, 6))
    ax.imshow(img)
    if label_path.exists():
        with open(label_path) as f:
            for line in f.readlines():
                cls, cx, cy, bw, bh = map(float, line.strip().split())
                x = (cx - bw / 2) * w
                y = (cy - bh / 2) * h
                color = COLORS[int(cls) % len(COLORS)]
                rect = patches.Rectangle((x, y), bw * w, bh * h,
                                         linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                ax.text(x, y - 5, class_names[int(cls)], color=color, fontsize=9,
                        bbox=dict(facecolor='white', alpha=0.6, pad=1))
    ax.axis('off')
    plt.tight_layout()
    plt.show()

train_imgs = list((Path(DATASET_PATH) / 'train' / 'images').glob('*'))[:4]
for img_path in train_imgs:
    label_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    plot_yolo_sample(img_path, label_path, data_config['names'])

## 6. Entrenar YOLOv8s

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data=str(data_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    project=f'{DRIVE_PATH}/Resultados_Pruebas',
    name='yolov8s_caries',
    exist_ok=True,
    plots=True,
    save=True,
    cache=True
)

print('Entrenamiento completado')
print(f'Mejor modelo en: {DRIVE_PATH}/Resultados_Pruebas/yolov8s_caries/weights/best.pt')

## 7. Evaluar el modelo en test set

In [ ]:
best_model_path = f'{DRIVE_PATH}/Resultados_Pruebas/yolov8s_caries/weights/best.pt'
best_model = YOLO(best_model_path)

metrics = best_model.val(data=str(data_yaml), split='test')

print('\n=== Métricas en Test Set ===')
print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

## 8. Graficar métricas de entrenamiento

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = f'{DRIVE_PATH}/Resultados_Pruebas/yolov8s_caries/results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Métricas de Entrenamiento YOLOv8s - Detección de Caries', fontsize=14)

metrics_to_plot = [
    ('train/box_loss',       'Train Box Loss'),
    ('train/cls_loss',       'Train Class Loss'),
    ('val/box_loss',         'Val Box Loss'),
    ('metrics/precision(B)', 'Precision'),
    ('metrics/recall(B)',    'Recall'),
    ('metrics/mAP50(B)',     'mAP@50'),
]

for ax, (col, title) in zip(axes.flatten(), metrics_to_plot):
    if col in df.columns:
        ax.plot(df['epoch'], df[col])
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/Resultados_Pruebas/training_metrics.png', dpi=150)
plt.show()

## 9. Inferencia de prueba en imágenes del test set

In [ ]:
import glob
import matplotlib.pyplot as plt

test_images = glob.glob(f'{DATASET_PATH}/test/images/*')[:4]

fig, axes = plt.subplots(1, len(test_images), figsize=(16, 5))
fig.suptitle('Predicciones del Modelo', fontsize=13)

for ax, img_path in zip(axes, test_images):
    result = best_model.predict(img_path, conf=0.25, verbose=False)[0]
    annotated = result.plot()
    ax.imshow(annotated[:, :, ::-1])
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/Resultados_Pruebas/sample_predictions.png', dpi=150)
plt.show()

## 10. Copiar modelo final a Drive

In [ ]:
import shutil

dest = f'{DRIVE_PATH}/models/best_caries.pt'
shutil.copy(best_model_path, dest)
print(f'Modelo guardado en: {dest}')